# UNIVERSIDADE FEDERAL FLUMINENSE

**TCE11616 - Introdução ao Aprendizado Profundo**

**Prof.: Diogo Ferreira de Lima Silva**

---
# AULA 08 — REDES CONVOLUCIONAIS
---

Até agora, para usar uma imagem em uma rede, nós a **achatávamos** em uma fila de números com `nn.Flatten()`. Isso funciona, mas joga fora uma informação preciosa: **quais pixels são vizinhos**. Uma borda, um canto, a pata de um animal... tudo isso é formado por pixels próximos.

As **redes convolucionais** (CNNs, do inglês *Convolutional Neural Networks*) foram criadas para aproveitar essa vizinhança.

Elas estão por trás da maioria das aplicações de visão computacional: inspeção de peças, leitura de placas, diagnóstico por imagem.

> **Dica:** no Colab, use *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU*. Em CPU, a aula inteira leva uns 3 minutos.

## 0. Importando as bibliotecas

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)      # fixa a semente: os resultados ficam iguais a cada execução

# cores usadas nos gráficos
LARANJA, AZUL, VERDE, VERMELHO, CINZA = "#E08D2B", "#1F7A8C", "#2E7D4F", "#B3151A", "#5B6472"

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

---
## 1. A ideia: um filtro que varre a imagem
---

Pense em uma **lupa** pequena, de $3 \times 3$ pixels, que passa por cada posição da imagem. Em cada posição, ela multiplica os 9 pixels que está vendo por 9 pesos e soma tudo, produzindo **um** número. O conjunto desses números forma uma nova imagem.

Esses 9 pesos são o **filtro**. Dependendo dos valores, o filtro destaca coisas diferentes.

Por exemplo, este filtro responde forte onde há uma **borda vertical** (claro de um lado, escuro do outro):

```
-1   0   1
-1   0   1
-1   0   1
```

Vamos testar em uma imagem do **Fashion-MNIST** (imagens $28 \times 28$ de roupas e calçados)

In [ ]:
# ToTensor: converte cada imagem em um tensor (canais, altura, largura), com valores entre 0 e 1
treino = datasets.FashionMNIST(root="dados", train=True,  download=True, transform=transforms.ToTensor())
teste  = datasets.FashionMNIST(root="dados", train=False, download=True, transform=transforms.ToTensor())
nomes = treino.classes        # os nomes das 10 classes

imagem, rotulo = treino[1]    # o segundo exemplo de treino (a contagem começa em 0)
print("formato da imagem:", imagem.shape, "| classe:", nomes[rotulo])

Em PyTorch, a camada de convolução é a `nn.Conv2d`. Aqui criamos uma com **2 filtros** e colocamos os pesos à mão: um para bordas verticais e outro para bordas horizontais.

Normalmente a rede **aprende** os pesos sozinha; aqui escolhemos à mão só para entender.

In [ ]:
# 1 canal de entrada (cinza) e 2 de saída (um por filtro). padding=1 mantém o tamanho 28 x 28.
# bias=False: sem viés, para o resultado depender só dos pesos que vamos escolher.
conv = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, padding=1, bias=False)

vertical = torch.tensor([[-1., 0., 1.],
                         [-1., 0., 1.],
                         [-1., 0., 1.]])
horizontal = vertical.T          # o mesmo filtro, "deitado"

# Trocamos os pesos sorteados pelos nossos filtros.
# conv.weight tem formato (canais de saída, canais de entrada, altura, largura).
# O no_grad é necessário porque o PyTorch não permite alterar parâmetros diretamente sem ele.
with torch.no_grad():
    conv.weight[0, 0] = vertical        # canal de saída 0, canal de entrada 0
    conv.weight[1, 0] = horizontal      # canal de saída 1, canal de entrada 0

# Aqui o no_grad evita calcular gradientes, pois só queremos aplicar os filtros.
with torch.no_grad():
    saida = conv(imagem.unsqueeze(0))     # unsqueeze(0): cria um "lote" com 1 imagem

print("entrada:", imagem.unsqueeze(0).shape)
print("saída  :", saida.shape)           # 1 imagem, 2 canais (um por filtro), 28 x 28

In [ ]:
# imagem[0]: o único canal da imagem. saida[0, c]: primeira imagem do lote, canal c.
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 3, 1); plt.imshow(imagem[0], cmap="gray_r");   plt.title("original"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(saida[0, 0], cmap="RdBu");   plt.title("filtro de bordas verticais"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(saida[0, 1], cmap="RdBu");   plt.title("filtro de bordas horizontais"); plt.axis("off")
plt.show()

Cada filtro produz um **mapa** que acende (vermelho ou azul) onde encontra o padrão que procura. Repare em três propriedades importantes:

- **cada saída depende só de uma vizinhança pequena** (3 × 3), e não da imagem inteira;
- **o mesmo filtro é usado em todas as posições**: uma borda no canto é detectada da mesma forma que uma borda no centro;
- por isso, um filtro tem **pouquíssimos parâmetros** (9 pesos), não importa o tamanho da imagem.

Uma CNN tem dezenas de filtros como esses, e **aprende** os pesos de cada um durante o treino.

---
## 2. As peças de uma CNN e os formatos
---

Uma CNN típica alterna duas peças:

| Peça | O que faz | Formato: entrada → saída |
|---|---|---|
| `nn.Conv2d(1, 16, kernel_size=3, padding=1)` | aplica 16 filtros 3 × 3 | `(N, 1, 28, 28)` → `(N, 16, 28, 28)` |
| `nn.MaxPool2d(2)` | reduz a imagem pela metade, guardando o maior valor de cada bloco 2 × 2 | `(N, 16, 28, 28)` → `(N, 16, 14, 14)` |

- O `padding=1` acrescenta uma borda de zeros em volta da imagem, para que a saída continue com $28 \times 28$.
- O **número de canais** cresce (1 → 16 → 32: mais tipos de padrão), enquanto o **tamanho** da imagem diminui (28 → 14 → 7).

No final, achatamos com `nn.Flatten()` e usamos camadas `nn.Linear` comuns para decidir a classe.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),    # 16 filtros sobre a imagem
    nn.MaxPool2d(2),                                          # reduz a imagem pela metade
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),   # 32 filtros sobre os 16 mapas anteriores
    nn.MaxPool2d(2),
    nn.Flatten(),                                             # transforma os mapas em um vetor
    nn.Linear(32 * 7 * 7, 64),
    nn.ReLU(),
    nn.Linear(64, 10),                                        # uma saída para cada classe
)

Vamos passar um lote de 64 imagens **camada por camada** e imprimir o formato depois de cada uma. Esse é o melhor jeito de entender (e de depurar) uma CNN.

In [ ]:
lote = torch.zeros(64, 1, 28, 28)     # um lote "falso", só para ver os formatos
print(f"{'entrada':<12} {tuple(lote.shape)}")

x = lote
for camada in cnn:          # um nn.Sequential pode ser percorrido camada por camada
    x = camada(x)           # a saída de uma camada é a entrada da seguinte
    print(f"{camada.__class__.__name__:<12} {tuple(x.shape)}")

Repare de onde vem o `32 * 7 * 7` da primeira `nn.Linear`: depois do segundo `MaxPool2d`, cada imagem virou **32 canais de 7 × 7**, que o `Flatten` transforma em $32 \times 7 \times 7 = 1568$ números.

> Se você mudar alguma camada e errar esse número, o PyTorch acusa um erro de formato (`mat1 and mat2 shapes cannot be multiplied`). Imprimir os formatos como acima resolve a dúvida.

---
## 3. CNN × rede comum no Fashion-MNIST
---

Vamos comparar a CNN com uma rede comum, totalmente conectada (MLP), que tem um número **parecido** de parâmetros.

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),                # 28 x 28 pixels viram uma fila de 784 números
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10),
)

def conta_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())     # numel: quantos números há em cada tensor

print("parâmetros da MLP:", conta_parametros(mlp))
print("parâmetros da CNN:", conta_parametros(cnn))

Repare onde estão os parâmetros da CNN: as duas convoluções somam só **4.800** (160 + 4.640), enquanto a primeira `nn.Linear` sozinha tem **100.416**.

As funções de treino e de avaliação são as mesmas da Aula 04 (MNIST), só que agora guardadas em funções.

Elas também enviam os dados para a GPU, se houver uma.

In [ ]:
# usa a GPU, se houver uma; senão, a CPU
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("usando:", dispositivo)

carregador_treino = DataLoader(treino, batch_size=64, shuffle=True)     # shuffle: nova ordem a cada época
carregador_teste  = DataLoader(teste, batch_size=1000)


def acuracia(modelo, carregador):
    modelo.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for imagens, rotulos in carregador:
            imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)    # dados e modelo no mesmo dispositivo
            # argmax(dim=1): a classe com a maior pontuação em cada imagem
            acertos += (modelo(imagens).argmax(dim=1) == rotulos).sum().item()
            total += len(rotulos)
    return acertos / total


def treinar(modelo, carregador_treino, carregador_teste, epocas=5):
    modelo.to(dispositivo)                 # leva os parâmetros para a GPU, se houver
    perda_fn = nn.CrossEntropyLoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
    historico = []
    for epoca in range(epocas):
        modelo.train()
        for imagens, rotulos in carregador_treino:
            imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
            perda = perda_fn(modelo(imagens), rotulos)
            otimizador.zero_grad()         # apaga os gradientes do lote anterior
            perda.backward()               # calcula os novos gradientes
            otimizador.step()              # ajusta os parâmetros
        acc = acuracia(modelo, carregador_teste)
        historico.append(acc)
        print(f"época {epoca + 1}: acurácia no teste = {acc:.2%}")
    return historico

Para simplificar, acompanhamos direto o teste, sem escolher nada com base nele. Em um trabalho real, como no projeto, as escolhas são feitas na validação.

> Treinar a MLP leva poucos segundos; a CNN, cerca de 40 segundos em CPU (a convolução faz muito mais contas por imagem).

In [ ]:
# Mesma semente nos dois treinos: os dois modelos recebem os lotes na mesma ordem.
print("=== MLP ===")
torch.manual_seed(42)
hist_mlp = treinar(mlp, carregador_treino, carregador_teste)

print("\n=== CNN ===")
torch.manual_seed(42)
hist_cnn = treinar(cnn, carregador_treino, carregador_teste)

In [ ]:
epocas = range(1, 6)       # as épocas 1 a 5
plt.figure(figsize=(7, 4))
plt.plot(epocas, hist_mlp, "o-", color=LARANJA, linewidth=2, label="MLP")
plt.plot(epocas, hist_cnn, "o-", color=VERDE, linewidth=2, label="CNN")
plt.xlabel("época"); plt.ylabel("acurácia no teste"); plt.xticks(epocas)
plt.title("Fashion-MNIST: rede comum × convolucional"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Com praticamente o **mesmo número de parâmetros**, a CNN acerta mais.

A diferença é **como** os parâmetros são usados: em vez de ligar cada pixel a cada neurônio, a CNN usa filtros pequenos que procuram padrões locais em toda a imagem.

---
## 4. O que a CNN aprendeu?
---

A primeira camada da CNN tem 16 filtros de $3 \times 3$. Vamos olhar os pesos aprendidos:

In [ ]:
# cnn[0] é a primeira Conv2d. detach() separa os pesos do cálculo de gradientes,
# e cpu() os traz da GPU, se for o caso: só assim o matplotlib consegue desenhá-los.
filtros = cnn[0].weight.detach().cpu()      # formato (16, 1, 3, 3)
print("formato dos filtros:", filtros.shape)

plt.figure(figsize=(10, 2.5))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(filtros[i, 0], cmap="RdBu")   # filtro i, canal de entrada 0 (o único)
    plt.axis("off")
plt.suptitle("Os 16 filtros aprendidos pela primeira camada")
plt.show()

Alguns filtros lembram detectores de **bordas** em várias direções, como os que escolhemos à mão na Seção 1, e outros são mais difíceis de interpretar. Ninguém programou esses pesos: eles vieram do treino.

Vejamos agora a saída desses filtros (os **mapas de características**) para uma imagem:

In [ ]:
imagem, rotulo = teste[0]        # a primeira imagem de teste
# Aplicamos só as duas primeiras peças da rede, sem passar pelas demais.
with torch.no_grad():
    mapas = cnn[1](cnn[0](imagem.unsqueeze(0).to(dispositivo))).cpu()   # Conv2d + ReLU

plt.figure(figsize=(12, 3.2))
plt.subplot(2, 9, 1); plt.imshow(imagem[0], cmap="gray_r"); plt.title(nomes[rotulo], fontsize=8); plt.axis("off")
for i in range(16):
    plt.subplot(2, 9, i + 2 + (i >= 8))      # pula o início da segunda linha, para alinhar os mapas
    plt.imshow(mapas[0, i], cmap="viridis"); plt.axis("off")
plt.show()

Cada mapa destaca uma parte diferente da peça: contornos de um lado, do outro, regiões claras, regiões escuras... As camadas seguintes combinam esses mapas em padrões cada vez mais complexos (cantos, texturas, partes de objetos).

---
## 5. Por que a CNN funciona melhor? Um experimento
---

Vamos **embaralhar os pixels** de todas as imagens, sempre com a **mesma** ordem sorteada. Para nós, as imagens viram ruído. Mas toda a informação continua lá: só mudou a posição dos pixels.

In [ ]:
torch.manual_seed(0)
ordem = torch.randperm(784)          # uma ordem aleatória fixa para os 784 pixels

# Achata a imagem em 784 números, muda a ordem e volta ao formato (1, 28, 28).
def embaralha(img):
    return img.reshape(-1)[ordem].reshape(1, 28, 28)

# Compose: aplica as transformações em sequência. Lambda: usa a nossa função como transformação.
transf_embaralhada = transforms.Compose([transforms.ToTensor(), transforms.Lambda(embaralha)])
treino_emb = datasets.FashionMNIST(root="dados", train=True,  download=True, transform=transf_embaralhada)
teste_emb  = datasets.FashionMNIST(root="dados", train=False, download=True, transform=transf_embaralhada)

plt.figure(figsize=(10, 2.5))
# teste[i][0][0]: exemplo i, a imagem (e não o rótulo), canal 0
for i in range(5):
    plt.subplot(2, 5, i + 1);     plt.imshow(teste[i][0][0], cmap="gray_r");     plt.axis("off")
    plt.subplot(2, 5, i + 6);     plt.imshow(teste_emb[i][0][0], cmap="gray_r"); plt.axis("off")
plt.suptitle("em cima: originais | embaixo: pixels embaralhados")
plt.show()

Treinamos uma MLP e uma CNN **novas** nas imagens embaralhadas.

> Esta célula leva cerca de 1 minuto em CPU.

In [ ]:
carr_treino_emb = DataLoader(treino_emb, batch_size=64, shuffle=True)
carr_teste_emb  = DataLoader(teste_emb, batch_size=1000)

# Modelos novos, com a mesma arquitetura de antes, treinados do zero.
torch.manual_seed(42)
mlp_emb = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
print("=== MLP, pixels embaralhados ===")
hist_mlp_emb = treinar(mlp_emb, carr_treino_emb, carr_teste_emb)

torch.manual_seed(42)
cnn_emb = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10),
)
print("\n=== CNN, pixels embaralhados ===")
hist_cnn_emb = treinar(cnn_emb, carr_treino_emb, carr_teste_emb)

In [ ]:
# hist[-1]: a acurácia na última época
print(f"{'':<6} {'originais':>10} {'embaralhadas':>13}")
print(f"{'MLP':<6} {hist_mlp[-1]:>10.1%} {hist_mlp_emb[-1]:>13.1%}")
print(f"{'CNN':<6} {hist_cnn[-1]:>10.1%} {hist_cnn_emb[-1]:>13.1%}")

- A **MLP** praticamente não se importa com o embaralhamento: para ela, a imagem sempre foi só uma fila de 784 números, sem noção de vizinhança.
- A **CNN** perde a vantagem: seus filtros 3 × 3 procuram padrões entre pixels **vizinhos**, e depois do embaralhamento os vizinhos não têm mais relação entre si.

A CNN não fica inútil (ainda acerta 86,9%): a ordem sorteada é sempre a mesma, então cada filtro continua combinando os mesmos pixels em todas as imagens, e as camadas `nn.Linear` aprendem o que cada posição significa. Ela só deixa de ter a vantagem da vizinhança.

Ou seja: a CNN é melhor em imagens **porque** aproveita a estrutura espacial. Essa é a sua grande ideia.

---
## 6. Vendo as previsões
---

Título verde: acerto. Vermelho: erro.

In [ ]:
imagens, rotulos = next(iter(carregador_teste))      # pega só o primeiro lote
cnn.eval()
with torch.no_grad():
    previstos = cnn(imagens.to(dispositivo)).argmax(dim=1).cpu()

plt.figure(figsize=(14, 5))
for i in range(18):
    plt.subplot(3, 6, i + 1)
    plt.imshow(imagens[i][0], cmap="gray_r"); plt.axis("off")
    cor = VERDE if previstos[i] == rotulos[i] else VERMELHO
    plt.title(nomes[previstos[i]], color=cor, fontsize=9)
plt.tight_layout(); plt.show()

---
## Exercícios
---

**1.** Na Seção 1, crie um filtro que detecte bordas na **diagonal**. Dica: experimente valores como `[[0, 1, 2], [-1, 0, 1], [-2, -1, 0]]`.

**2.** Acrescente um terceiro bloco à CNN: `nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)`. Use o laço da Seção 2 para descobrir o novo número de entradas da primeira `nn.Linear`. (Dica: $7 / 2$ arredondado para baixo é 3.)

**3.** Troque `kernel_size=3` por `kernel_size=5` (e `padding=2`) nas duas convoluções. A acurácia muda? E o número de parâmetros?

**4.** Treine a CNN no **MNIST** (dígitos) em vez do Fashion-MNIST. Qual foi a acurácia encontrada?